In [3]:
!pip install pymupdf4llm

  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.8/19.8 MB 3.6 MB/s eta 0:00:06
   ---- ----------------------------------- 2.1/19.8 MB 5.0 MB/s eta 0:00:04
   ------- -------------------------------- 3.7/19.8 MB 6.0 MB/s eta 0:00:03
   ---------- ----------------------------- 5.2/19.8 MB 6.4 MB/s eta 0:00:03
   -------------- ------------------------- 7.1/19.8 MB 6.8 MB/s eta 0:00:02
   ------------------ --------------------- 8.9/19.8 MB 7.1 MB/s eta 0:00:02
   -------------------- ------------------- 10.2/19.8 MB 7.1 MB/s eta 0:00:02
   ----------------------- ---------------- 11.8/19.8 MB 7.1 MB/s eta 0:00:02
   --------------------------- ------------ 13.4/19.8 MB 7.2 MB/s eta 0:00:01
   ------------------------------ --------- 15.2/19.8 MB 7.2 MB/s eta 0:00:01
   --------------------------------- ------ 16.8/19.8 MB 7.3 MB/s eta 0:00:01
   

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-cpu 2.21.0 requires absl-py>=1.0.0, which is not installed.
tensorflow-cpu 2.21.0 requires astunparse>=1.6.0, which is not installed.
tensorflow-cpu 2.21.0 requires gast!=0.5.0,!=0.5.1,!=0.5.2,>=0.2.1, which is not installed.
tensorflow-cpu 2.21.0 requires google_pasta>=0.1.1, which is not installed.
tensorflow-cpu 2.21.0 requires grpcio<2.0,>=1.24.3, which is not installed.
tensorflow-cpu 2.21.0 requires h5py<3.15.0,>=3.11.0, which is not installed.
tensorflow-cpu 2.21.0 requires keras>=3.12.0, which is not installed.
tensorflow-cpu 2.21.0 requires libclang>=13.0.0, which is not installed.
tensorflow-cpu 2.21.0 requires ml_dtypes<1.0.0,>=0.5.1, which is not installed.
tensorflow-cpu 2.21.0 requires opt_einsum>=2.3.2, which is not installed.
tensorflow-cpu 2.21.0 requires termcolor>=1.1.0, which is not 

In [9]:
import pymupdf4llm


In [8]:
pip install langchain-text-splitters langchain-core

  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/561.6 kB ? eta -:--:--
   ---------------------------------------- 561.6/561.6 kB 4.7 MB/s  0:00:00
   ---------------------------------------- 0.0/675.2 kB ? eta -:--:--
   ------------------------------- -------- 524.3/675.2 kB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 675.2/675.2 kB 2.0 MB/s  0:00:00
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)

   -- -------------------------------------  1/15 [xxhash]
   ----- ----------------------------------  2/15 [websockets]
   ----- ----------------------------------  2/15 [websockets]
   ----- ----------------------------------  2/15 [websockets]
   -------- -------------------------------  3/15 [uuid-utils]
   ------------- --------------------------  5/15 [

In [10]:
from langchain_text_splitters import MarkdownHeaderTextSplitter,RecursiveCharacterTextSplitter,Language
from langchain_core.documents import Document

In [13]:
from pathlib import Path
# Path containing your 6 policy PDFs
DATA_DIR = Path("data/policies")
pdf_files = list(DATA_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF(s) to process.\n")

# Step 1: Define Markdown Header hierarchy
headers_to_split_on = [("#", "Header 1"),("##", "Header 2"),("###", "Header 3"),]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on,
                                               strip_headers=False  # Keeps section titles inside the text
                                               )

# Step 2: Define Markdown-aware character splitter (large chunk size to protect tables)
text_splitter = RecursiveCharacterTextSplitter.from_language(language=Language.MARKDOWN,
                                                             chunk_size=1200,      # Keeps multi-line tables intact
                                                             chunk_overlap=150
                                                             )

Found 5 PDF(s) to process.



In [14]:
all_documents = []

# Process each PDF file step-by-step
for idx, pdf_path in enumerate(pdf_files, start=1):
    insurer_name = pdf_path.stem.replace("_", " ").title()
    print(f"[{idx}/{len(pdf_files)}] Parsing: {pdf_path.name}...")

    # Extract Markdown page-by-page to retain accurate page numbers
    page_data = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)

    for page_dict in page_data:
        page_num = page_dict["metadata"]["page_number"]  # 1-based page number
        page_text = page_dict["text"]

        if not page_text.strip():
            continue

        # A. Split by Markdown headers first
        header_splits = markdown_splitter.split_text(page_text)

        # B. Sub-split large text blocks
        final_splits = text_splitter.split_documents(header_splits)

        # C. Attach file and page metadata
        for doc in final_splits:
            doc.metadata.update({
                "source": pdf_path.name,
                "insurer": insurer_name,
                "page": page_num
            })
            all_documents.append(doc)

print(f"\n✅ Processing Complete! Total Chunks Extracted: {len(all_documents)}")

[1/5] Parsing: Star_Comprehensive_Insurance_Policy.pdf...
[2/5] Parsing: Star_Diabetes_Safe_Insurance_Policy.pdf...
[3/5] Parsing: Star_Senior_Citizens_Red_Carpet_Health_Insurance_Policy.pdf...
[4/5] Parsing: Star_Women_Care_Insurance_Policy.pdf...
[5/5] Parsing: Star_Young_Extra_Protect_Add_On.pdf...

✅ Processing Complete! Total Chunks Extracted: 291


In [2]:
!pip install langchain-community

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB 2.6 MB/s eta 0:00:01
   -------------------------- ------------- 1.6/2.4 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 2.5 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 3.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 2.8 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 4.0 MB/s  0:00:00
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB

In [17]:
!pip install langchain_huggingface

   ---------------------------------------- 0.0/771.9 kB ? eta -:--:--
   ------------- -------------------------- 262.1/771.9 kB ? eta -:--:--
   --------------------------- ------------ 524.3/771.9 kB 1.6 MB/s eta 0:00:01
   --------------------------- ------------ 524.3/771.9 kB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 771.9/771.9 kB 1.2 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 1.8 MB/s eta 0:00:02
   ------- -------------------------------- 0.8/4.0 MB 1.7 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/4.0 MB 1.5 MB/s eta 0:00:02
   --------------------- ------------------ 2.1/4.0 MB 2.1 MB/s eta 0:00:01
   ---------------------------- ----------- 2.9/4.0 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------  3.9/4.0 MB 2.8 MB/s eta 0:00:01
   ----------------------

In [18]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [19]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

In [ ]:
from langchain_community.vectorstores import FAISS

print("Generating vector embeddings and saving local FAISS index...")

# Vectorize all chunks and store in FAISS database
vectorstore = FAISS.from_documents(all_documents, embeddings)

# Save index to local folder
vectorstore.save_local("faiss_index")

print("🎉 SUCCESS! 'faiss_index/' directory created with index.faiss and index.pkl!")